# PCA Factor Regression — Monthly Hazelnut Price

**No production data.** Extract latent factors from 16 financial/commodity return series, regress monthly hazelnut USD price returns on the factors that matter.

- **n = 207 months** (2005–2026, with known TOBB gaps dropped)
- y = monthly log return of TOBB all-exchange VWAP in USD/kg
- Features = monthly log returns of equities, commodity futures, FX

In [ ]:
import sys, warnings
sys.path.insert(0, '.')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from pca_regression import run_all, build_feature_matrix, run_pca
from tobb_price_regression import _ols

plt.rcParams.update({'figure.dpi': 130, 'font.size': 9})

out       = run_all()
sub       = out['sub']
feat_cols = out['feat_cols']
pca_out   = out['pca_out']
results   = out['results']
best      = out['best_model']

print(f"n={len(sub)}  ({sub['month'].iloc[0]} – {sub['month'].iloc[-1]})")
print(f"{len(feat_cols)} features: {feat_cols}")

## 1 · Feature universe — bivariate correlations with hazelnut price return

In [ ]:
y     = sub['ret_vwap_usd'].values
ones  = np.ones(len(y))

corrs = sub[feat_cols].corrwith(sub['ret_vwap_usd']).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 4))
colors = ['steelblue' if v > 0 else 'crimson' for v in corrs]
ax.barh(corrs.index, corrs.values, color=colors)
ax.axvline(0, color='black', lw=0.8)
ax.set_xlabel('Pearson correlation with monthly hazelnut USD return')
ax.set_title('Bivariate correlations — all 16 features')
plt.tight_layout()
plt.show()

## 2 · PCA — variance explained and component loadings

PC1 = broad commodity / risk-on factor (Bunge, SPY, corn, soy, MDLZ).
PC3 = USD strength factor (DXY dominates at +0.70, Hershey, inverse oil/gold).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Scree plot
ax = axes[0]
ve = pca_out['var_explained']
ax.bar(range(1, len(ve)+1), ve, color='steelblue')
ax.set_xlabel('Component')
ax.set_ylabel('Variance explained (%)')
ax.set_title('Scree plot')

# PC1 loadings
ax = axes[1]
ld1 = pca_out['loadings']['PC1'].sort_values()
colors1 = ['steelblue' if v > 0 else 'crimson' for v in ld1]
ax.barh(ld1.index, ld1.values, color=colors1)
ax.axvline(0, color='black', lw=0.5)
ax.set_title('PC1 loadings\n(commodity / risk-on cycle)')
ax.set_xlabel('loading')

# PC3 loadings
ax = axes[2]
ld3 = pca_out['loadings']['PC3'].sort_values()
colors3 = ['steelblue' if v > 0 else 'crimson' for v in ld3]
ax.barh(ld3.index, ld3.values, color=colors3)
ax.axvline(0, color='black', lw=0.5)
ax.set_title('PC3 loadings\n(USD strength / Hershey factor)')
ax.set_xlabel('loading')

plt.tight_layout()
plt.show()

## 3 · Which PCs explain hazelnut price?

In [ ]:
print(results.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
sig_pcs = results[results['PC_p'].notna() & (results['PC_p'] < 0.05)]
all_pcs = results[results['PC_p'].notna()]
colors  = ['steelblue' if p < 0.05 else 'lightgray'
           for p in all_pcs['PC_p'].values]
ax.bar(all_pcs['model'], all_pcs['R2'], color=colors)
ax.axhline(0, color='black', lw=0.5)
ax.set_ylabel('R² (bivariate)')
ax.set_title('R² of each PC vs hazelnut price return (blue = p<0.05)')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()

## 4 · Best model: PC1 + PC3

**R²=0.338, adj-R²=0.332, n=207.** Both components p<0.001.

PC1 (commodity cycle) and PC3 (USD/Hershey) are orthogonal by construction — no multicollinearity.

In [ ]:
print(f"R2={best.rsquared:.3f}  adj-R2={best.rsquared_adj:.3f}  AIC={best.aic:.1f}  n={best.n}")
print()
print(pd.DataFrame({'coef': best.params, 't': best.tvalues, 'p': best.pvalues}).round(3).to_string())

In [ ]:
scores = pca_out['scores']
fitted = best.fitted
resid  = best.resid

sub_copy = sub.copy()
sub_copy['date'] = pd.to_datetime(sub_copy['month'])

fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True)

ax = axes[0]
ax.plot(sub_copy['date'], sub_copy['ret_vwap_usd'], lw=0.9, color='steelblue', alpha=0.7, label='actual')
ax.plot(sub_copy['date'], fitted, lw=1.2, color='crimson', label='PC1+PC3 fitted')
ax.axhline(0, color='black', lw=0.4)
ax.set_ylabel('log return')
ax.set_title('Monthly hazelnut USD return: actual vs PC1+PC3 fitted')
ax.legend(fontsize=8)

ax = axes[1]
ax.bar(sub_copy['date'], resid, width=20,
       color=['crimson' if r < 0 else 'steelblue' for r in resid], alpha=0.8)
ax.axhline(0, color='black', lw=0.4)
ax.set_ylabel('residual')
ax.set_title('Residuals')

import matplotlib.dates as mdates
for ax in axes:
    ax.xaxis.set_major_locator(mdates.YearLocator(2))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.tight_layout()
plt.show()

## 5 · Interpretation

| Component | What it is | Proxy asset |
|-----------|-----------|-------------|
| **PC1** | Global commodity / risk-on cycle. Bunge, SPY, corn, soy, MDLZ all load positively. VIX loads negatively. | SPY or GSG |
| **PC3** | USD strength + domestic consumer resilience. DXY dominates (+0.70), Hershey positive, oil/gold negative. | UUP (dollar ETF) + HSY |

**What this means for basket construction**: hazelnut USD prices co-move with broad commodity markets and with USD strength. A long position in commodity ETF (PC1) + dollar ETF (PC3) would partially hedge hazelnut price exposure — but only ~33% of variance, the rest is supply-driven (frost, production shocks) that financial markets don't price in advance.